# Phase 1: Data Understanding and Exploration
**Goal:** understand the bank's customer data and what drives a customer saying *yes* to the term deposit, **before** building any model.

Sections: 1. Setup, 2. First look, 3. Data dictionary, 4. Data quality, 5. Target, 6. Features vs target, 7. Data leakage (`duration`), 8. Correlations, 9. Insights, 10. Decisions for preprocessing

## 1. Setup

In [ ]:
import sys
sys.path.append("..")  # lets the notebook import from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

from src.config import TARGET, FIGURES_DIR
from src.data import load_data

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(name):
    """Save the current figure to reports/figures (used later in the report) and show it."""
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{name}.png", dpi=120)
    plt.show()

## 2. First look at the data

In [ ]:
df = load_data()
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Data dictionary (business meaning)

| Group | Column | Meaning |
|---|---|---|
| **Client** | `age` | Age in years |
| | `job`, `marital`, `education` | Job type, marital status, education level |
| | `default`, `housing`, `loan` | Has credit in default / housing loan / personal loan |
| **This campaign** | `contact` | Contact channel (cellular / telephone) |
| | `month`, `day_of_week` | When the last contact happened |
| | `duration` | Length of the last call in seconds (**only known after the call**) |
| | `campaign` | Number of contacts made to this client in this campaign |
| **Past campaigns** | `pdays` | Days since the client was last contacted in a previous campaign (`999` = never) |
| | `previous` | Number of contacts before this campaign |
| | `poutcome` | Outcome of the previous campaign |
| **Economy** | `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed` | Quarterly/monthly macro indicators (employment variation, consumer price index, consumer confidence, 3-month Euribor rate, number employed) |
| **Target** | `y` | Did the client subscribe to the term deposit? (yes / no) |

## 4. Data quality checks

In [ ]:
print("Missing values (NaN):", int(df.isna().sum().sum()))
print("Duplicate rows:      ", int(df.duplicated().sum()))

No real `NaN`s, but in this dataset **missing values are hidden as the text `"unknown"`**. Let's count them.

In [ ]:
categorical_cols = df.select_dtypes(exclude="number").columns.drop(TARGET)
unknown_counts = (df[categorical_cols] == "unknown").sum()
unknown_counts = unknown_counts[unknown_counts > 0].sort_values(ascending=False)

pd.DataFrame({
    "unknown_count": unknown_counts,
    "unknown_percent": (unknown_counts / len(df) * 100).round(1),
})

In [ ]:
# `default` is almost useless as a "yes/no" column
df["default"].value_counts()

`pdays = 999` is not a real number of days. It is a **placeholder** for "never contacted before".

In [ ]:
print("pdays == 999:", f"{(df.pdays == 999).mean():.1%} of rows")
print("\nReal pdays values (clients contacted before):")
print(df.loc[df.pdays != 999, "pdays"].describe().round(1))

In [ ]:
# Other unusual values
print("Calls with duration = 0 s:", int((df.duration == 0).sum()))
print("Clients contacted more than 10 times in this campaign:", int((df.campaign > 10).sum()), "| max =", df.campaign.max())
print("Age range:", df.age.min(), "to", df.age.max())

## 5. Target variable

In [ ]:
df["target"] = (df[TARGET] == "yes").astype(int)   # 1 = subscribed
overall = df["target"].mean()

print(df[TARGET].value_counts())
print(f"\nConversion rate: {overall:.1%}")
print(f"A model that ALWAYS predicts 'no' would be {1 - overall:.1%} accurate, and useless.")

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(data=df, x=TARGET, order=["no", "yes"], ax=ax)
ax.set_title("Target distribution")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
save_fig("target_distribution")

## 6. Features vs target

A helper that shows the **conversion rate** per group (with the number of calls `n`). The red line is the overall rate.

In [ ]:
def conversion_table(col):
    return df.groupby(col, observed=True)["target"].agg(rate="mean", calls="count")


def plot_conversion(col, title=None, order=None):
    """Horizontal bar chart of conversion rate per group."""
    t = conversion_table(col)
    t = t.reindex(order) if order is not None else t.sort_values("rate", ascending=False)

    fig, ax = plt.subplots(figsize=(8, max(3, 0.42 * len(t))))
    ax.barh(t.index.astype(str), t["rate"] * 100, color="#4C72B0")
    ax.axvline(overall * 100, color="red", ls="--", label=f"overall {overall:.1%}")
    for i, (rate, n) in enumerate(zip(t["rate"], t["calls"])):
        ax.text(rate * 100 + 0.5, i, f"{rate:.1%}  (n={n:,})", va="center", fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, t["rate"].max() * 100 * 1.35)
    ax.set_xlabel("Conversion rate (%)")
    ax.set_title(title or f"Conversion rate by {col}")
    ax.legend(loc="lower right")
    save_fig(f"conversion_by_{col}")
    return t

### 6.1 Customer profile: job, age, marital status

In [ ]:
_ = plot_conversion("job")

In [ ]:
df["age_group"] = pd.cut(df.age, [0, 25, 35, 45, 55, 65, 100],
                         labels=["17-25", "26-35", "36-45", "46-55", "56-65", "66+"])
_ = plot_conversion("age_group", order=["17-25", "26-35", "36-45", "46-55", "56-65", "66+"])

In [ ]:
_ = plot_conversion("marital")

### 6.2 How we contact: channel, month, number of calls

In [ ]:
_ = plot_conversion("contact")

In [ ]:
months = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
_ = plot_conversion("month", order=months)

In [ ]:
df["campaign_group"] = pd.cut(df.campaign, [0, 1, 2, 3, 5, 10, 100],
                              labels=["1", "2", "3", "4-5", "6-10", "11+"])
_ = plot_conversion("campaign_group", title="Conversion rate by number of contacts in this campaign",
                    order=["1", "2", "3", "4-5", "6-10", "11+"])

### 6.3 History with the bank: previous campaigns

In [ ]:
_ = plot_conversion("poutcome", title="Conversion rate by outcome of the previous campaign")

In [ ]:
df["contacted_before"] = np.where(df.pdays == 999, "never contacted", "contacted before")
_ = plot_conversion("contacted_before")

### 6.4 The economy: Euribor 3-month rate

In [ ]:
df["euribor_group"] = pd.cut(df.euribor3m, [0, 1, 2, 4, 6], labels=["<=1", "1-2", "2-4", ">4"])
_ = plot_conversion("euribor_group", title="Conversion rate by 3-month Euribor rate (%)",
                    order=["<=1", "1-2", "2-4", ">4"])

## 7. Data leakage check: `duration`

`duration` is the length of the call. **We only know it after the call has happened**, but the business wants to predict *before* calling. Using it would be cheating. Let's see how strongly it is linked to the target.

In [ ]:
print(df.groupby(TARGET)["duration"].agg(["mean", "median"]).round(0))
print(f"\nROC-AUC of duration ALONE: {roc_auc_score(df['target'], df['duration']):.3f}")

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.boxplot(data=df, x=TARGET, y="duration", order=["no", "yes"], showfliers=False, ax=ax)
ax.set_title("Call duration by outcome")
save_fig("duration_leakage")

One feature, which is unknown before the call, already reaches an AUC of about 0.82. A model using it would look great in testing and be **useless in production**. We will **drop `duration`**. The dataset's own documentation recommends the same for a realistic model.

## 8. Correlations between numeric features

In [ ]:
num_cols = ["age", "campaign", "previous", "emp.var.rate", "cons.price.idx",
            "cons.conf.idx", "euribor3m", "nr.employed", "target"]
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix")
save_fig("correlation_matrix")

`emp.var.rate`, `euribor3m` and `nr.employed` are almost the same signal (correlations 0.91 to 0.97). That matters for Logistic Regression (unstable coefficients) and for interpreting feature importance.

## 9. Insights (observation, then business implication)

1. **Imbalanced target.** Only **11.3%** of clients subscribe (4,640 of 41,188). Always predicting "no" gives 88.7% accuracy.
   *Implication:* accuracy is misleading. We evaluate with PR-AUC, precision, recall and handle the imbalance (class weights, threshold tuning).

2. **`duration` is leakage.** Successful calls last on average **553 s vs 221 s**, and duration alone gives an AUC of about 0.82, but it is only known after the call.
   *Implication:* drop it, otherwise the model would look excellent offline and fail in real use.

3. **Past success is the strongest customer signal.** Clients whose previous campaign was a *success* convert at **65.1%**, versus **8.8%** for those never contacted before. People contacted before (`pdays != 999`) convert at **63.8%** vs **9.3%**.
   *Implication:* re-contacting past successes should be the first priority, which is cheap and high yield.

4. **Economic conditions matter a lot.** When the 3-month Euribor is at most 1%, conversion is **45.7%**; above 4% it is only **4.8%**.
   *Implication:* the model partly learns the economic period rather than customer behavior. It will degrade if rates change (limitation, and a reason to monitor drift).

5. **Cellular beats telephone.** **14.7%** vs **5.2%** conversion.
   *Implication:* prefer mobile contact where possible.

6. **Calling the same client repeatedly has diminishing returns.** Conversion falls from **13.0%** (1 contact) to **3.1%** (11+ contacts).
   *Implication:* cap the number of attempts per client (about 3 to 5) and redirect effort to new prospects.

7. **Some segments convert far above average.** Students **31.4%**, retired **25.2%**, ages 66+ **46.8%** and 17-25 **20.9%**, while blue-collar is only **6.9%**.
   *Implication:* useful for targeting, but age and marital status are sensitive attributes, so we must check fairness (Phase 5).

8. **Month effects are confounded with volume.** March, September, October and December convert at 44-51% but have few calls (about 2,000 in total), while May has 13,769 calls at only 6.4%.
   *Implication:* the bank concentrated effort in its worst month. The few high-rate months are small samples, so be careful not to over-trust them.

9. **Data quality issues.** "unknown" appears in 6 columns (`default` 20.9%, `education` 4.2%, `housing`/`loan` 2.4%, `job` 0.8%, `marital` 0.2%), `default = yes` has only 3 rows, there are 12 duplicate rows, and `pdays = 999` is a placeholder.
   *Implication:* keep "unknown" as its own category, remove duplicates, and engineer a proper flag from `pdays`.

10. **The data is ordered in time.** It starts in May and ends in November (2008 to 2010), covering the financial crisis.
    *Implication:* a random split is fine for this exercise, but production would need a time-based split, and results may not generalize to other economic periods.

## 10. Decisions carried into preprocessing (Phase 2)

| Issue | Decision |
|---|---|
| `duration` leakage | Drop it |
| 12 duplicate rows | Drop them |
| `"unknown"` categories | Keep as their own category (it can be informative, and there are no real NaNs) |
| `pdays = 999` | Create `was_contacted_before` flag, set pdays to 0 for never-contacted |
| Numeric features | Standardize (needed for Logistic Regression) |
| Categorical features | One-hot encode, ignore unseen categories at inference |
| Imbalance (11.3%) | `class_weight` / `scale_pos_weight` plus threshold tuning; metric: PR-AUC |
| Split | Stratified 70 / 15 / 15; test set touched once at the end |